In [1]:
# ==========================================
# Project : SeeIt - Crop Leaf Disease Classification
# Notebook : 03 - MobileNetV2 Training
# Purpose : Build and prepare MobileNetV2 using Transfer Learning
# ==========================================



In [2]:
# ==========================================
# Import Required Libraries
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models, transforms
from torchvision.datasets import ImageFolder

from torch.utils.data import DataLoader, random_split

from pathlib import Path

import matplotlib.pyplot as plt

In [3]:
# ==========================================
# Define Dataset Path
# ==========================================

BASE_DIR = Path(r"C:\Users\bhall\OneDrive\Desktop\project\data")

color_dir = BASE_DIR / "color"

print("Dataset Path:", color_dir)
print("Folder Exists:", color_dir.exists())

Dataset Path: C:\Users\bhall\OneDrive\Desktop\project\data\color
Folder Exists: True


In [4]:
# ==========================================
# Load PlantVillage Dataset
# ==========================================

color_dataset = ImageFolder(
    root=color_dir,
    transform=transforms.ToTensor()
)

print("Dataset Loaded Successfully!")
print("Total Images:", len(color_dataset))
print("Total Classes:", len(color_dataset.classes))

Dataset Loaded Successfully!
Total Images: 54297
Total Classes: 38


In [5]:
# ==========================================
# Explore One Sample from the Dataset
# ==========================================

image, label = color_dataset[0]

print("Image Shape:", image.shape)
print("Label:", label)
print("Class Name:", color_dataset.classes[label])

Image Shape: torch.Size([3, 256, 256])
Label: 0
Class Name: Apple___Apple_scab


In [6]:
# ==========================================
# Create Image Transform Pipeline
# ==========================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Training and Validation transforms created successfully!")

Training and Validation transforms created successfully!


In [7]:
# ==========================================
# Load Dataset with Transform Pipeline
# ==========================================

full_dataset = ImageFolder(
    root=color_dir,
    transform=train_transform
)

print("Dataset Loaded Successfully!")
print("Total Images:", len(full_dataset))
print("Total Classes:", len(full_dataset.classes))

Dataset Loaded Successfully!
Total Images: 54297
Total Classes: 38


In [8]:
# ==========================================
# Split Dataset into Train and Validation
# ==========================================

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

print("Training Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

Training Images: 43437
Validation Images: 10860


In [9]:
# ==========================================
# Create DataLoaders
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 1358
Validation batches: 340


In [10]:
# ==========================================
# Load Pretrained MobileNetV2
# ==========================================

mobilenet = models.mobilenet_v2(weights="DEFAULT")

print("MobileNetV2 loaded successfully!")

MobileNetV2 loaded successfully!


In [11]:
# ==========================================
# Replace the Final Classification Layer
# ==========================================

mobilenet.classifier[1] = nn.Linear(
    in_features=1280,
    out_features=38
)

print("Final classifier replaced successfully!")

Final classifier replaced successfully!


In [12]:
# ==========================================
# Freeze Feature Extractor Layers
# ==========================================

for param in mobilenet.features.parameters():
    param.requires_grad = False

print("Feature extractor frozen successfully!")

Feature extractor frozen successfully!


In [13]:
# ==========================================
# Move Model to Device
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mobilenet = mobilenet.to(device)

print("Device:", device)

Device: cpu


In [14]:
# ==========================================
# Define Loss Function and Optimizer
# ==========================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mobilenet.classifier.parameters(),
    lr=0.001
)

print("Loss function and optimizer are ready!")

Loss function and optimizer are ready!


In [15]:
# ==========================================
# Training Loop Starts Here
# ==========================================

# Number of times the model will see the complete training dataset
num_epochs = 5

# Start training
for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Put the model in training mode
    mobilenet.train()


Epoch 1/5

Epoch 2/5

Epoch 3/5

Epoch 4/5

Epoch 5/5


In [16]:
# Loop through every batch in the training dataset
for images, labels in train_loader:

    # Move images and labels to the selected device (CPU/GPU)
    images = images.to(device)
    labels = labels.to(device)

    # Stop after the first batch (only for testing)
    break

print("One batch loaded successfully!")
print("Image Batch Shape:", images.shape)
print("Label Batch Shape:", labels.shape)

One batch loaded successfully!
Image Batch Shape: torch.Size([32, 3, 224, 224])
Label Batch Shape: torch.Size([32])


In [17]:
# ==========================================
# Forward Pass
# ==========================================

# Pass the images through MobileNetV2
outputs = mobilenet(images)

print("Output Shape:", outputs.shape)

Output Shape: torch.Size([32, 38])


In [18]:
# ==========================================
# Calculate Loss
# ==========================================

loss = criterion(outputs, labels)

print("Loss:", loss.item())

Loss: 3.644198417663574


In [19]:
# ==========================================
# Clear Previous Gradients
# ==========================================

optimizer.zero_grad()

In [20]:
# ==========================================
# Backward Pass
# ==========================================

loss.backward()

print("Backward pass completed!")

Backward pass completed!


In [21]:
# ==========================================
# Update Model Weights
# ==========================================

optimizer.step()

print("Weights updated successfully!")

Weights updated successfully!


In [22]:
# ==========================================
# Get Predicted Classes
# ==========================================

_, predicted = torch.max(outputs, 1)

print("Predicted Labels:")
print(predicted)

print("\nActual Labels:")
print(labels)

Predicted Labels:
tensor([23, 13,  1, 16, 36, 15, 23, 20, 37, 15, 26, 13, 26,  6,  3, 34,  6, 10,
         8, 15, 10, 10, 36, 16, 23, 37, 16,  0, 28, 13, 22, 37])

Actual Labels:
tensor([29, 12, 15, 24, 26, 30, 24, 30, 13, 32, 33, 31,  2, 25, 30, 15, 24, 24,
        16, 28, 24, 24, 29,  0, 15, 33, 35, 15, 19, 30, 35, 19])


In [23]:
# ==========================================
# Calculate Batch Accuracy
# ==========================================

correct = (predicted == labels).sum().item()

accuracy = correct / labels.size(0) * 100

print(f"Correct Predictions: {correct}/{labels.size(0)}")
print(f"Batch Accuracy: {accuracy:.2f}%")

Correct Predictions: 0/32
Batch Accuracy: 0.00%


In [24]:
# ==========================================
# Complete Training Loop
# ==========================================

num_epochs = 10                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

for epoch in range(num_epochs):

    mobilenet.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward Pass
        outputs = mobilenet(images)

        # Calculate Loss
        loss = criterion(outputs, labels)

        # Backward Pass
        loss.backward()

        # Update Weights
        optimizer.step()

        # Store Loss
        running_loss += loss.item()

        # Calculate Accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Loss: {epoch_loss:.4f}")
    print(f"Accuracy: {epoch_accuracy:.2f}%\n")

Epoch [1/10]
Loss: 0.6443
Accuracy: 86.00%

Epoch [2/10]
Loss: 0.2457
Accuracy: 93.47%

Epoch [3/10]
Loss: 0.1988
Accuracy: 94.24%

Epoch [4/10]
Loss: 0.1719
Accuracy: 94.74%

Epoch [5/10]
Loss: 0.1632
Accuracy: 94.91%

Epoch [6/10]
Loss: 0.1516
Accuracy: 95.21%

Epoch [7/10]
Loss: 0.1435
Accuracy: 95.43%

Epoch [8/10]
Loss: 0.1388
Accuracy: 95.40%

Epoch [9/10]
Loss: 0.1351
Accuracy: 95.52%

Epoch [10/10]
Loss: 0.1350
Accuracy: 95.44%



In [25]:
print(images.shape)
print(labels.shape)

torch.Size([13, 3, 224, 224])
torch.Size([13])


In [26]:
# ==========================================
# Validation Loop
# ==========================================

mobilenet.eval()

val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass
        outputs = mobilenet(images)

        # Calculate Loss
        loss = criterion(outputs, labels)

        val_loss += loss.item()

        # Predictions
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

validation_loss = val_loss / len(val_loader)
validation_accuracy = 100 * correct / total

print("Validation Completed!")
print(f"Validation Loss: {validation_loss:.4f}")
print(f"Validation Accuracy: {validation_accuracy:.2f}%")

Validation Completed!
Validation Loss: 0.0992
Validation Accuracy: 96.68%


In [30]:
# ==========================================
# Save the Trained Model
# ==========================================

torch.save(
    mobilenet.state_dict(),
    "mobilenet_plant_disease_new.pth"
)

print("Model saved successfully!")

Model saved successfully!
